# connectivity_lmm_raincloud.ipynb\n\n**Purpose:** Analyse longitudinal changes in inter-regional functional\nconnectivity (frontal × temporal) using a LMM and Mann-Whitney test.\n\n**Inputs:**\n- connstats_completo.xlsx — Fisher-z connectivity values exported by conectividade_funcional.m\n\n**Outputs:**\n- connectivity_results_hbo.xlsx — LMM coefficients and Mann-Whitney results\n- Raincloud and trajectory plots (PNG)\n\n**Model:** conexao_inter ~ group * session + conn_baseline + (1|subject) [REML, BFGS]\n- conexao_inter = mean Fisher-z over all 60 inter-ROI channel pairs per session\n- N = 121 obs (3 excluded: SUBJ_001, SUBJ_005, SUBJ_024 at session 4 — poor signal quality)\n- conn_baseline = connectivity at calibration session (sessao_num=0), as covariate\n\n**Library:** statsmodels 0.14.6 (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Functional Connectivity — Inter-ROI LMM + Raincloud
**Project SESI | Input: ConnStats_leitura.xlsx**

- Connectivity: Frontal × Temporal (Fisher-z, HbO)
- Session 0 = calibration (anchor), Sessions 1–9 = training
- Model: `conexao_inter ~ group * session + (1|subject)`
- Raincloud: Mann-Whitney on mean connectivity (sessions 1–9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu, linregress
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration

Excluir - 'S6-D5': ('P7-P9', 'L Inferior Temporal Gyrus (29%)', 'TEMPORAL'),

In [ ]:
# Update this path to match your local data directory
PATH_IN        = r'../data/connstats_completo.xlsx'
# Update this path to match your local data directory
PATH_OUT_EXCEL = r'../results/connectivity_results_st02.xlsx'
# Update this path to match your local data directory
PATH_OUT_TRAJ  = r'../results/connectivity_trajectory_st02.png'
# Update this path to match your local data directory
PATH_OUT_RAIN  = r'../results/connectivity_raincloud_st02.png'

# ROI channel lists
ROI_FRONTAL  = [f'S{s}-D{d}' for s, d in [(1,1),(2,1),(2,3),(1,2),(3,3),(3,1)]]
ROI_TEMPORAL = [f'S{s}-D{d}' for s, d in [(8,7),(8,4),(7,7),(7,4),(7,6),(6,5),(6,4),(5,5),(5,6),(5,4)]]
CHROMOPHORE = 'hbo'  # options: 'hbo' | 'StO2' | 'hbr'
PALETTE = {
    'acelerado'    : 'darkorange',
    'nao_acelerado': 'steelblue'
}
LABELS = {
    'acelerado'    : 'Accelerated',
    'nao_acelerado': 'Non-Accelerated'
}

## 2 — Load and prepare

In [ ]:
df = pd.read_excel(PATH_IN)

# Keep HbO only
df = df[(df['TypeOrigin'] == CHROMOPHORE) & (df['TypeDest'] == CHROMOPHORE)].copy()

# Fill NaN Z with 0
df['Z'] = df['Z'].fillna(0)

# Flag inter-ROI pairs: one channel in FRONTAL, other in TEMPORAL
df['in_frontal']  = df['channel_origin'].isin(ROI_FRONTAL)  | df['channel_dest'].isin(ROI_FRONTAL)
df['in_temporal'] = df['channel_origin'].isin(ROI_TEMPORAL) | df['channel_dest'].isin(ROI_TEMPORAL)
df['is_inter_roi'] = (
    (df['channel_origin'].isin(ROI_FRONTAL)  & df['channel_dest'].isin(ROI_TEMPORAL)) |
    (df['channel_origin'].isin(ROI_TEMPORAL) & df['channel_dest'].isin(ROI_FRONTAL))
)

df_inter = df[df['is_inter_roi']].copy()

# Sanity check
n_pairs = len(ROI_FRONTAL) * len(ROI_TEMPORAL)
print(f'Inter-ROI pairs per session: {n_pairs}  (expected {len(ROI_FRONTAL)}×{len(ROI_TEMPORAL)} = {n_pairs})')
print(f'Total inter-ROI rows: {len(df_inter)}')
print(f'Sessions: {sorted(df_inter["session"].unique())}')
print(f'Subjects: {df_inter["subject"].nunique()}')

## 3 — Aggregate: mean inter-ROI connectivity per subject × session

In [ ]:
# Mean Fisher-z across all Frontal×Temporal pairs
df_long = (
    df_inter
    .groupby(['subject', 'group', 'session'])['Z']
    .mean()
    .reset_index()
    .rename(columns={'Z': 'conexao_inter'})
)

# Back-transform to r for descriptive stats
df_long['r_mean'] = np.tanh(df_long['conexao_inter'])

# Set reference category
df_long['group'] = pd.Categorical(
    df_long['group'],
    categories=['nao_acelerado', 'acelerado'],
    ordered=False
)
df_long['subject'] = pd.Categorical(df_long['subject'])

print(f'Long format shape: {df_long.shape}  (expected ~{14*10} rows)')
print(f'Sessions included: {sorted(df_long["session"].unique())}')
print(f'\nDescriptive (r_mean) by group:')
print(df_long.groupby('group', observed=True)['r_mean'].describe().round(3))

In [ ]:
# Separate baseline (session 0) and training (sessions 1–9)
df_baseline = df_long[df_long['session'] == 0][['subject','conexao_inter']]\
    .rename(columns={'conexao_inter': 'conn_baseline'})

df_long = df_long[df_long['session'] >= 1].copy()

df_long = pd.merge(df_long, df_baseline, on='subject', how='left')

print(f'Training rows: {len(df_long)}  (expected ~{14*9} = 126)')
print(f'Missing conn_baseline: {df_long["conn_baseline"].isna().sum()}')
print(df_long[['subject','session','group','conexao_inter','conn_baseline']].head(3).to_string())

In [ ]:
# Exclude specific subject-session pairs with poor signal quality
# Verified in MATLAB: ac_04 signal quality insufficient for SUBJ_001, SUBJ_005, SUBJ_024
EXCLUDE_PAIRS = [
    ('SUBJ_001', 4),
    ('SUBJ_005', 4),
    ('SUBJ_024', 4),
]

before = len(df_long)
for subj, sess in EXCLUDE_PAIRS:
    mask = (df_long['subject'] == subj) & (df_long['session'] == sess)
    df_long = df_long[~mask]

after = len(df_long)
print(f'Rows removed: {before - after}  (expected {len(EXCLUDE_PAIRS)})')
print(f'Remaining rows: {after}')
print(f'\nVerification — session 4 non-accelerated:')
print(df_long[(df_long['session'] == 4) & 
              (df_long['group'] == 'nao_acelerado')]
      [['subject','session','conexao_inter']].to_string(index=False))

In [ ]:
# Winsorize conexao_inter at 5th and 95th percentile
#p5  = df_long['conexao_inter'].quantile(0.05)
#p95 = df_long['conexao_inter'].quantile(0.95)
#
#print(f'Winsorization thresholds:')
#print(f'  5th percentile : {p5:.4f}')
#print(f'  95th percentile: {p95:.4f}')

#n_low  = (df_long['conexao_inter'] < p5).sum()
#n_high = (df_long['conexao_inter'] > p95).sum()
#print(f'  Values clipped low : {n_low}')
#print(f'  Values clipped high: {n_high}')

#df_long['conexao_inter'] = df_long['conexao_inter'].clip(lower=p5, upper=p95)

#print(f'\nAfter winsorization:')
#print(f'  Min: {df_long["conexao_inter"].min():.4f}')
#print(f'  Max: {df_long["conexao_inter"].max():.4f}')
#print(f'  Mean: {df_long["conexao_inter"].mean():.4f}')

## 4 — LMM: `conexao_inter ~ group * session + (1|subject)`
Session 0 = calibration (anchor/intercept)

In [ ]:
lmm = smf.mixedlm(
    "conexao_inter ~ group * session + conn_baseline",
    data=df_long,
    groups=df_long['subject']
)
res = lmm.fit(reml=True, method='bfgs')
print(res.summary())

# Clean results table
fe_index = res.fe_params.index
ci       = res.conf_int().loc[fe_index]
df_lmm = pd.DataFrame({
    'parameter': fe_index,
    'coef'     : res.fe_params.values,
    'se'       : res.bse.loc[fe_index].values,
    'z'        : (res.fe_params / res.bse.loc[fe_index]).values,
    'pvalue'   : res.pvalues.loc[fe_index].values,
    'ci_lower' : ci[0].values,
    'ci_upper' : ci[1].values,
}).assign(significant=lambda d: d['pvalue'] < 0.05)

print('\nKey result — group × session interaction:')
key = df_lmm[df_lmm['parameter'].str.contains('session', case=False)]
print(key[['parameter','coef','pvalue','ci_lower','ci_upper','significant']].to_string(index=False))

## 5 — Trajectory plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

traj = (
    df_long.groupby(['session', 'group'], observed=True)['conexao_inter']
    .agg(['mean', 'sem'])
    .reset_index()
)

for grp in ['acelerado', 'nao_acelerado']:
    sub   = traj[traj['group'] == grp].sort_values('session')
    color = PALETTE[grp]
    label = LABELS[grp]

    ax.plot(sub['session'], sub['mean'],
            marker='o', color=color, label=label, linewidth=2)
    ax.fill_between(
        sub['session'],
        sub['mean'] - sub['sem'],
        sub['mean'] + sub['sem'],
        alpha=0.2, color=color
    )
    # Regression line
    m, b, *_ = linregress(sub['session'], sub['mean'])
    x_line = np.linspace(sub['session'].min(), sub['session'].max(), 100)
    ax.plot(x_line, m * x_line + b,
            linestyle='--', color=color, linewidth=1.2, alpha=0.7)

# Annotate LMM interaction
try:
    inter_row = df_lmm[df_lmm['parameter'].str.contains(':session|session.*acelerado', regex=True)].iloc[0]
    p_str  = f"p={inter_row['pvalue']:.3f}" if inter_row['pvalue'] >= 0.001 else 'p<0.001'
    sig    = '★' if inter_row['significant'] else 'n.s.'
    ax.annotate(
        
    )
except Exception:
    pass

# Mark calibration boundary
ax.axvline(0.5, linestyle=':', color='grey', linewidth=1.2, alpha=0.6)
ax.text(0.55, ax.get_ylim()[0], 'calibration | training →',
        fontsize=8, color='grey', va='bottom')

ax.set_xticks(range(0, 10))
ax.set_xticklabels(['Calib'] + [str(i) for i in range(1, 10)])
ax.set_xlabel('Session', fontsize=11)
ax.set_ylabel('Inter-ROI Connectivity (mean Fisher-z ± SEM)', fontsize=11)
ax.set_title('Frontal–Temporal Connectivity Trajectory by Group',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(PATH_OUT_TRAJ, dpi=300)
print(f'-> Figure saved: {PATH_OUT_TRAJ}')
plt.show()

In [ ]:
# Check session 4 non-accelerated subjects
s4 = df_long[
    (df_long['session'] == 4) & 
    (df_long['group'] == 'nao_acelerado')
][['subject','session','conexao_inter']].sort_values('conexao_inter')

print('Non-accelerated — Session 4:')
print(s4.to_string(index=False))

# Compare with sessions 3 and 5 for same subjects
s3_5 = df_long[
    (df_long['session'].isin([3,5])) & 
    (df_long['group'] == 'nao_acelerado')
][['subject','session','conexao_inter']].sort_values(['subject','session'])

print('\nSame group — Sessions 3 and 5:')
print(s3_5.to_string(index=False))

## 6 — Mann-Whitney + Raincloud (sessions 1–9 only)

In [ ]:
# Mean connectivity per subject over training sessions only
df_train = df_long[df_long['session'] >= 1].copy()
df_subj = (
    df_train.groupby(['subject', 'group'], observed=True)['conexao_inter']
    .mean()
    .reset_index()
    .rename(columns={'conexao_inter': 'conn_mean'})
)

# Mann-Whitney
acc  = df_subj[df_subj['group'] == 'acelerado']['conn_mean'].values
ctrl = df_subj[df_subj['group'] == 'nao_acelerado']['conn_mean'].values
U, p = mannwhitneyu(acc, ctrl, alternative='two-sided')
r    = 1 - (2 * U) / (len(acc) * len(ctrl))

def es_label(r):
    r = abs(r)
    if r < 0.3: return 'small'
    if r < 0.5: return 'medium'
    return 'large'

p_str  = f"p={p:.3f}" if p >= 0.001 else 'p<0.001'
sig    = '✱' if p < 0.05 else 'n.s.'
print(f'Mann-Whitney: U={U}, {p_str}, r={r:.3f} ({es_label(r)}) {sig}')

# ── Raincloud plot ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
groups  = ['acelerado', 'nao_acelerado']
xpos    = {'acelerado': 0, 'nao_acelerado': 1}

for grp in groups:
    data  = df_subj[df_subj['group'] == grp]['conn_mean'].values
    x     = xpos[grp]
    color = PALETTE[grp]

    # Violin
    vp = ax.violinplot(data, positions=[x], widths=0.4,
                       showmeans=False, showmedians=False, showextrema=False)
    for body in vp['bodies']:
        m = np.mean(body.get_paths()[0].vertices[:, 0])
        if grp == 'acelerado':
            body.get_paths()[0].vertices[:, 0] = np.clip(
                body.get_paths()[0].vertices[:, 0], m, np.inf)
        else:
            body.get_paths()[0].vertices[:, 0] = np.clip(
                body.get_paths()[0].vertices[:, 0], -np.inf, m)
        body.set_facecolor(color)
        body.set_alpha(0.3)
        body.set_edgecolor(color)

    # Boxplot
    ax.boxplot(data, positions=[x], widths=0.12,
               patch_artist=True, showfliers=False,
               medianprops=dict(color='black', linewidth=2),
               boxprops=dict(facecolor=color, alpha=0.6),
               whiskerprops=dict(color=color),
               capprops=dict(color=color))

    # Strip
    jitter = np.random.uniform(-0.06, 0.06, size=len(data))
    ax.scatter(x + jitter, data,
               color=color, alpha=0.9, s=60, zorder=5,
               edgecolors='black', linewidths=0.5)

# Stats annotation
ax.annotate(
    f"{sig}  {p_str}\nr={r:.3f} ({es_label(r)})",
    xy=(0.5, 0.97), xycoords='axes fraction',
    ha='center', va='top', fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='grey', alpha=0.8)
)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Accelerated', 'Non-Accelerated'], fontsize=11)
ax.set_ylabel('Mean Inter-ROI Connectivity (Fisher-z)', fontsize=11)
ax.set_title('Frontal–Temporal Connectivity by Group\n(Mean across training sessions)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(PATH_OUT_RAIN, dpi=300)
print(f'-> Figure saved: {PATH_OUT_RAIN}')
plt.show()

## 7 — Export results

In [ ]:
mw_df = pd.DataFrame([{
    'U': U, 'p_value': round(p, 4),
    'effect_size_r': round(r, 3),
    'magnitude': es_label(r),
    'significant': p < 0.05
}])

with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_lmm.to_excel(writer,  sheet_name='LMM_results',    index=False)
    df_long.to_excel(writer, sheet_name='connectivity_long', index=False)
    df_subj.to_excel(writer, sheet_name='subject_means',  index=False)
    mw_df.to_excel(writer,   sheet_name='mannwhitney',    index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print('   Sheets: LMM_results | connectivity_long | subject_means | mannwhitney')